# Structural-information results

This notebook aggregates the saved test_metrics.csv files for the frozen structural-information probes. It does not load embeddings, checkpoints, manifests, datasets, or models. Run it from the repository root in the main environment.

Metrics are calculated per target on the held-out test set. For the nine binary functional-group targets, ROC-AUC measures whether positives are ranked above negatives and average precision summarizes the precision-recall curve across probability thresholds; neither uses a fixed classification threshold. For the eight continuous molecular descriptors, R² measures the fraction of test-set variation explained by the prediction (1 is perfect, 0 matches predicting the test-set mean).

It writes per-target metrics, macro summaries, and mean ranks under results/structural_information/analysis/. Interpretation belongs in the benchmark documentation, not here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "results/structural_information").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root")

RESULTS_DIR = PROJECT_ROOT / "results/structural_information"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
METRICS = {"functional": ("roc_auc", "average_precision"), "descriptors": ("r2", "mae")}
PRIMARY_METRICS = {"roc_auc", "average_precision", "r2"}
LABELS = {
    "nmrpeak": "NMRPeak-R", "nmrtrans": "NMRTrans", "ultranmr": "UltraNMR",
    "nmrsolver": "NMR-Solver", "fomonmr-pre-shifts": "FoMoNMR pre / shifts",
    "fomonmr-post-shifts": "FoMoNMR post / shifts", "fomonmr-post-rich": "FoMoNMR post / rich",
    "fomonmr-post-unimol-shifts": "FoMoNMR post-UniMol / shifts",
    "fomonmr-post-unimol-rich": "FoMoNMR post-UniMol / rich",
}
ORDER = list(LABELS)


## Load and validate the saved probe metrics

The published comparison contains nine representations, nine functional targets, eight descriptor targets, linear seed 42, and MLP seeds 13, 42, and 73. These checks ensure that only directly comparable metrics are aggregated.

In [2]:
def load_metrics():
    required = {"probe", "seed", "target", "support", "positives", "prevalence",
                "roc_auc", "average_precision", "r2", "mae"}
    tables = []
    for path in sorted(RESULTS_DIR.glob("*/test_metrics.csv")):
        table = pd.read_csv(path)
        missing = required.difference(table.columns)
        if missing:
            raise ValueError(f"{path} is missing {sorted(missing)}")
        table["representation"] = path.parent.name
        tables.append(table)
    if not tables:
        raise FileNotFoundError(f"No test_metrics.csv below {RESULTS_DIR}")
    return pd.concat(tables, ignore_index=True)


def validate(raw):
    if set(raw.representation) != set(ORDER):
        raise ValueError("Expected exactly the declared benchmark representations")
    if raw.duplicated(["representation", "probe", "seed", "target"]).any():
        raise ValueError("Duplicate representation/probe/seed/target rows")
    for name, rows in raw.groupby("representation"):
        if len(rows) != 68:
            raise ValueError(f"{name}: expected 68 rows, found {len(rows)}")
        if set(rows.loc[rows.probe == "linear", "seed"]) != {42}:
            raise ValueError(f"{name}: expected linear seed 42")
        if set(rows.loc[rows.probe == "mlp", "seed"]) != {13, 42, 73}:
            raise ValueError(f"{name}: expected MLP seeds 13, 42, 73")
    if raw.loc[raw.roc_auc.notna(), "target"].nunique() != 9:
        raise ValueError("Expected 9 functional targets")
    if raw.loc[raw.r2.notna(), "target"].nunique() != 8:
        raise ValueError("Expected 8 descriptor targets")
    for column in ("support", "positives", "prevalence"):
        if raw.groupby("target")[column].nunique(dropna=True).gt(1).any():
            raise ValueError(f"Target {column} differs between representations")


raw = load_metrics()
validate(raw)
display(raw.groupby("representation", as_index=False).size().assign(representation=lambda t: t.representation.map(LABELS)))


,representation,size
0,FoMoNMR post / rich,68
1,FoMoNMR post / shifts,68
2,FoMoNMR post-UniMol / rich,68
3,FoMoNMR post-UniMol / shifts,68
4,FoMoNMR pre / shifts,68
5,NMRPeak-R,68
6,NMR-Solver,68
7,NMRTrans,68
8,UltraNMR,68


## Aggregate results

MLP macro values are averaged across targets within each seed before calculating mean and sample standard deviation across seeds. MAE remains target-level because descriptor targets have different units.

In [3]:
id_columns = ["representation", "probe", "seed", "target", "support", "positives", "prevalence"]
long = pd.concat([
    raw.loc[raw[metrics[0]].notna(), id_columns + list(metrics)].melt(
        id_vars=id_columns, value_vars=list(metrics), var_name="metric", value_name="value"
    ).assign(task=task)
    for task, metrics in METRICS.items()
], ignore_index=True)

per_target = long.groupby(["representation", "probe", "task", "target", "metric"], as_index=False).agg(
    mean=("value", "mean"),
    std=("value", lambda values: values.std(ddof=1) if len(values) > 1 else np.nan),
    n_seeds=("value", "size"),
    support=("support", "first"), positives=("positives", "first"), prevalence=("prevalence", "first"),
)

macro_per_seed = long[long.metric.isin(PRIMARY_METRICS)].groupby(
    ["representation", "probe", "task", "seed", "metric"], as_index=False
).agg(value=("value", "mean"))
macro_summary = macro_per_seed.groupby(["representation", "probe", "task", "metric"], as_index=False).agg(
    mean=("value", "mean"),
    std=("value", lambda values: values.std(ddof=1) if len(values) > 1 else np.nan),
    n_seeds=("value", "size"),
)


## Macro summaries

Linear has one seed. MLP values are mean ± sample standard deviation over the three probe seeds.

In [4]:
def format_score(row):
    return f"{row['mean']:.4f}" if row['n_seeds'] == 1 else f"{row['mean']:.4f} ± {row['std']:.4f}"

def show_summary(probe):
    table = macro_summary[macro_summary.probe == probe].copy()
    table["score"] = table.apply(format_score, axis=1)
    table["representation"] = table.representation.map(LABELS)
    return table.pivot(index="representation", columns="metric", values="score").reindex([LABELS[name] for name in ORDER])

display(show_summary("linear"))
display(show_summary("mlp"))


metric,average_precision,r2,roc_auc
representation,,,
NMRPeak-R,0.4942,0.5676,0.8107
NMRTrans,0.5614,0.4792,0.8549
UltraNMR,0.5417,0.4762,0.8331
NMR-Solver,0.4908,0.4313,0.8134
FoMoNMR pre / shifts,0.5236,0.4596,0.8256
FoMoNMR post / shifts,0.5176,0.4513,0.8224
FoMoNMR post / rich,0.5765,0.5082,0.8528
FoMoNMR post-UniMol / shifts,0.4903,0.4296,0.8075
FoMoNMR post-UniMol / rich,0.5425,0.4906,0.8351


metric,average_precision,r2,roc_auc
representation,,,
NMRPeak-R,0.5279 ± 0.0006,0.6267 ± 0.0006,0.8293 ± 0.0008
NMRTrans,0.6736 ± 0.0010,0.6302 ± 0.0006,0.8983 ± 0.0002
UltraNMR,0.5904 ± 0.0003,0.5474 ± 0.0007,0.8625 ± 0.0001
NMR-Solver,0.5779 ± 0.0019,0.5419 ± 0.0009,0.8563 ± 0.0001
FoMoNMR pre / shifts,0.5695 ± 0.0001,0.5148 ± 0.0003,0.8532 ± 0.0005
FoMoNMR post / shifts,0.5657 ± 0.0010,0.5073 ± 0.0002,0.8506 ± 0.0006
FoMoNMR post / rich,0.6316 ± 0.0008,0.5801 ± 0.0002,0.8828 ± 0.0002
FoMoNMR post-UniMol / shifts,0.5427 ± 0.0010,0.4890 ± 0.0002,0.8375 ± 0.0006
FoMoNMR post-UniMol / rich,0.6023 ± 0.0002,0.5721 ± 0.0002,0.8675 ± 0.0007


## Mean rank across target-level primary metrics

For each probe, every representation is ranked on 34 target-level metrics: ROC-AUC and average precision for nine functional targets, plus R² and MAE for eight descriptors. Lower MAE is better; all other metrics are higher-is-better.

In [5]:
rank_input = per_target.copy()
rank_input["rank"] = np.nan
for metric in ("roc_auc", "average_precision", "r2", "mae"):
    mask = rank_input.metric == metric
    rank_input.loc[mask, "rank"] = rank_input.loc[mask].groupby(
        ["probe", "task", "target", "metric"]
    )["mean"].rank(method="average", ascending=(metric == "mae"))

mean_rank = rank_input.groupby(["representation", "probe"], as_index=False).agg(
    mean_rank=("rank", "mean"), n_primary_metrics=("rank", "size")
)
mean_rank["representation"] = mean_rank.representation.map(LABELS)
mean_rank = mean_rank.sort_values(["probe", "mean_rank"]).reset_index(drop=True)
display(mean_rank)


,representation,probe,mean_rank,n_primary_metrics
0,FoMoNMR post / rich,linear,2.264706,34
1,NMRTrans,linear,3.117647,34
2,UltraNMR,linear,4.088235,34
3,FoMoNMR post-UniMol / rich,linear,4.294118,34
4,NMRPeak-R,linear,4.823529,34
5,FoMoNMR pre / shifts,linear,5.058824,34
6,FoMoNMR post / shifts,linear,5.941176,34
7,NMR-Solver,linear,7.264706,34
8,FoMoNMR post-UniMol / shifts,linear,8.147059,34
9,NMRTrans,mlp,1.264706,34


## Save final numeric tables

FoMoNMR variants remain ordinary rows in the shared comparison. This notebook deliberately does not calculate deltas or draw conclusions.

In [7]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
per_target.to_csv(ANALYSIS_DIR / "per_target.csv", index=False)
macro_summary.to_csv(ANALYSIS_DIR / "macro_summary.csv", index=False)
mean_rank.to_csv(ANALYSIS_DIR / "mean_rank.csv", index=False)
print(f"Wrote final CSV files to {ANALYSIS_DIR}")


Wrote final CSV files to /mnt/filesystem-test/nmr/results/structural_information/analysis
